# Rescan check: same space, different scans

ScanNet ids look like `sceneXXXX_YY`: `XXXX` is the space, `YY` the scan index, so
`scene0000_00` and `scene0000_01` are two scans of the same room. 3RScan has the same
idea (a reference scan plus rescans of one environment), but its ids are UUIDs, so the
grouping comes from the official `3RScan.json` metadata.

This notebook shows, for a few groups, each scan's caption and a few evenly spaced frames
side by side, to judge how similar rescans of one space really are. It also counts distinct
spaces, which is what a contrastive "same scene = positive" group id should be built on.

**Adjust the path config in the first code cell before running.**

In [ ]:
import json, random
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# --- path config -- adjust these to your actual layout ---
ROOT = Path("/glob/g01-cache/pf/Yushuo/vjepa201")
SCANNET_CAP = ROOT / "source_data/sceneverse/scannet_scene_cap.json"
SCANNET_POSED = ROOT / "source_data/scannet/posed_images"
LEO_3RSCAN = ROOT / "source_data/leo_annotations/annotations/alignment/scene_caption"
THREERSCAN_ROOT = ROOT / "source_data/3rscan"
SEED = 0
N_FRAMES = 4   # frames shown per scan

scannet_cap = json.load(open(SCANNET_CAP))
leo_3rscan = {}
for split in ("train", "val"):
    leo_3rscan.update(json.load(open(LEO_3RSCAN / f"3rscan_scenecap_{split}.json")))
print(f"scannet scans with captions: {len(scannet_cap)}   3rscan scans with captions: {len(leo_3rscan)}")


In [ ]:
def frames_for_scannet(scan_id):
    d = SCANNET_POSED / scan_id
    return sorted(p for p in d.glob("*.jpg") if "-annotated" not in p.stem)


def frames_for_3rscan(scan_id):
    # 3RScan color frames may be stored rotated (portrait sensor); shown as-is.
    return sorted((THREERSCAN_ROOT / scan_id / "sequence").glob("frame-*.color.jpg"))


def show_scans(scans, caption_of, frames_of, n_frames=N_FRAMES):
    for s in scans:
        frames = frames_of(s)
        print("=" * 100)
        print(f"{s}   ({len(frames)} frames)")
        print(caption_of(s)[:600])
        if not frames:
            continue
        idx = np.linspace(0, len(frames) - 1, n_frames).astype(int)
        fig, axes = plt.subplots(1, n_frames, figsize=(4 * n_frames, 3.2))
        for ax, i in zip(axes, idx):
            ax.imshow(Image.open(frames[i]))
            ax.set_title(frames[i].name, fontsize=8)
            ax.axis("off")
        plt.show()


## 1. ScanNet: group scans by space (`sceneXXXX`)

In [ ]:
def scannet_space_id(scan_id):
    return scan_id.split("_")[0]   # scene0000_01 -> scene0000


spaces = defaultdict(list)
for sid in sorted(scannet_cap):
    spaces[scannet_space_id(sid)].append(sid)

per_space = Counter(len(v) for v in spaces.values())
print(f"{len(scannet_cap)} scans -> {len(spaces)} distinct spaces")
print("scans per space:", dict(sorted(per_space.items())))


## 2. ScanNet: rescans of one space, side by side

Set `SPACE` (e.g. `"scene0000"`) to pin one, or leave `None` for a random multi-scan space.
Re-run with a different `SEED` to see other examples.

In [ ]:
rng = random.Random(SEED)
SPACE = None
multi = sorted(k for k, v in spaces.items() if len(v) > 1)
space = SPACE or rng.choice(multi)
print("space:", space, "->", spaces[space])
show_scans(spaces[space], lambda s: scannet_cap[s]["captions"][0], frames_for_scannet)


## 3. 3RScan: environment grouping from `3RScan.json`

The scripts under `dataset/` never downloaded this metadata file. The cell below looks for it
locally and tries to fetch it once (needs network). If that fails, download it manually from
`http://campar.in.tum.de/public_datasets/3RScan/3RScan.json` to the path printed.

In [ ]:
META_CANDIDATES = [THREERSCAN_ROOT / "3RScan.json", ROOT / "source_data/3rscan_meta/3RScan.json"]
META_URL = "http://campar.in.tum.de/public_datasets/3RScan/3RScan.json"

meta_path = next((p for p in META_CANDIDATES if p.exists()), None)
if meta_path is None:
    import urllib.request
    meta_path = META_CANDIDATES[0]
    try:
        urllib.request.urlretrieve(META_URL, meta_path)
        print("downloaded", meta_path)
    except Exception as e:
        print("could not fetch 3RScan.json:", e)
        print("download it manually to", meta_path)
        meta_path = None
else:
    print("found", meta_path)

if meta_path is not None:
    meta = json.load(open(meta_path))
    print(type(meta), len(meta))
    print(json.dumps(meta[0], indent=1)[:1000])


The next cell assumes each entry looks like `{"reference": <uuid>, "scans": [{"reference": <uuid>, ...}, ...]}`.
If it raises a `KeyError`, compare against the structure printed above and adjust.

In [ ]:
env_of = {}
for entry in meta:
    ref = entry["reference"]
    env_of[ref] = ref
    for s in entry.get("scans", []):
        env_of[s["reference"]] = ref

covered = sum(1 for sid in leo_3rscan if sid in env_of)
print(f"3RScan captioned scans found in metadata: {covered}/{len(leo_3rscan)}")

envs = defaultdict(list)
for sid in sorted(leo_3rscan):
    envs[env_of.get(sid, sid)].append(sid)

per_env = Counter(len(v) for v in envs.values())
print(f"{len(leo_3rscan)} scans -> {len(envs)} distinct environments")
print("scans per environment:", dict(sorted(per_env.items())))


## 4. 3RScan: rescans of one environment, side by side

3RScan rescans are recorded at different times, so small objects may have moved, appeared or
disappeared between them. Compare the captions and frames for what actually differs.

In [ ]:
rng = random.Random(SEED)
multi3 = sorted(k for k, v in envs.items() if len(v) > 1)
env = rng.choice(multi3)
print("environment:", env, "->", envs[env])
show_scans(envs[env], lambda s: leo_3rscan[s][0]["response"], frames_for_3rscan)


## 5. Summary and a reusable group id

In [ ]:
def group_id(scan_id):
    if scan_id.startswith("scene"):
        return "scannet/" + scannet_space_id(scan_id)
    return "3rscan/" + env_of.get(scan_id, scan_id)


print(f"ScanNet: {len(scannet_cap)} scans, {len(spaces)} spaces")
print(f"3RScan:  {len(leo_3rscan)} scans, {len(envs)} environments")
print(f"total:   {len(scannet_cap) + len(leo_3rscan)} scans, {len(spaces) + len(envs)} distinct spaces")
print()
for sid in [next(iter(scannet_cap)), next(iter(leo_3rscan))]:
    print(sid, "->", group_id(sid))
